# anomaly-vision
## Zero-Shot Surface Defect Detection
### PatchCore + Diffusion Refinement on MVTec AD

**Architecture overview**
- Teacher backbone (WideResNet50, pretrained ImageNet) extracts multi-scale patch features
- Memory bank built from normal images only — no defect labels needed
- Coreset subsampling keeps the bank tractable
- Diffusion prior partially reconstructs test images — defects "heal", residual map refines localisation
- Evaluation: Image AUROC · Pixel AUROC · PRO Score

---
> Runtime: **T4 GPU** required  →  `Runtime > Change runtime type > T4 GPU`


## 1 · Runtime check

In [7]:
import subprocess, sys

# Verify GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print("✓ GPU detected:", result.stdout.strip())
else:
    raise RuntimeError("No GPU found. Go to Runtime → Change runtime type → T4 GPU")

import torch
print(f"✓ PyTorch  : {torch.__version__}")
print(f"✓ CUDA     : {torch.version.cuda}")
print(f"✓ Device   : {torch.cuda.get_device_name(0)}")


✓ GPU detected: Tesla T4, 15360 MiB
✓ PyTorch  : 2.10.0+cu128
✓ CUDA     : 12.8
✓ Device   : Tesla T4


## 2 · Install dependencies

In [8]:
%%capture
!pip install timm>=0.9.12 diffusers>=0.25.0 scikit-image>=0.21.0 accelerate -q
print("✓ Dependencies installed")


## 3 · Clone repo & set working directory

In [9]:
import os

REPO_URL = "https://github.com/SankaVaas/anomaly-vision.git"  # ← update this
REPO_DIR = "/content/anomaly-vision"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(os.path.join(REPO_DIR, "src"))
print(f"✓ Working directory: {os.getcwd()}")


Already up to date.
✓ Working directory: /content/anomaly-vision/src


## 4 · Download MVTec AD dataset

MVTec AD is available via Kaggle.  
Set your Kaggle credentials once and the cell handles everything.


In [10]:
import os, zipfile, shutil
from pathlib import Path

DATA_DIR = "/content/anomaly-vision/data"
RAW_DIR  = f"{DATA_DIR}/raw"
PROC_DIR = f"{DATA_DIR}/processed"

os.makedirs(RAW_DIR, exist_ok=True)

# ── Configure Kaggle credentials ──
os.makedirs("/root/.kaggle", exist_ok=True)
if os.path.exists("/content/kaggle.json"):
    shutil.copy("/content/kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    print("✓ kaggle.json configured")
else:
    raise FileNotFoundError(
        "Upload your kaggle.json first:\n"
        "  Files panel (left sidebar) → Upload → select kaggle.json"
    )

# ── Download & unzip ──
DATASET_SLUG = "ipythonx/mvtec-ad"

raw_files = list(Path(RAW_DIR).iterdir()) if Path(RAW_DIR).exists() else []
if not raw_files:
    print("Downloading MVTec AD from Kaggle...")
    !kaggle datasets download -d {DATASET_SLUG} -p {RAW_DIR}
    print("Download complete. Extracting...")
else:
    print("✓ Raw files already present — skipping download")

# Find and extract zip
zips = list(Path(RAW_DIR).glob("*.zip"))
if zips:
    zip_path = str(zips[0])
    print(f"Extracting {zip_path} ...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(RAW_DIR)
    os.remove(zip_path)
    print("✓ Extraction done")

# ── Detect actual extracted structure ──
# MVTec can land as: raw/ (flat), raw/mvtec_anomaly_detection/, raw/mvtec/, etc.
KNOWN_CATEGORIES = {
    "bottle","cable","capsule","carpet","grid","hazelnut",
    "leather","metal_nut","pill","screw","tile",
    "toothbrush","transistor","wood","zipper"
}

def find_mvtec_root(search_dir):
    """Walk until we find a dir containing >= 5 known category subfolders."""
    for root, dirs, _ in os.walk(search_dir):
        if len(KNOWN_CATEGORIES.intersection(set(dirs))) >= 5:
            return root
    return None

mvtec_root = find_mvtec_root(RAW_DIR)

if mvtec_root is None:
    print("\n  Contents of RAW_DIR (debug):")
    for root, dirs, files in os.walk(RAW_DIR):
        depth = root.replace(RAW_DIR, "").count(os.sep)
        if depth > 2: continue
        print("  " * depth + os.path.basename(root) + "/")
    raise RuntimeError("Could not locate MVTec categories. Check tree above.")

print(f"✓ MVTec root detected: {mvtec_root}")

# ── Symlink raw root → processed ──
if os.path.islink(PROC_DIR): os.unlink(PROC_DIR)
if os.path.isdir(PROC_DIR):  shutil.rmtree(PROC_DIR)
os.symlink(mvtec_root, PROC_DIR)
print(f"✓ processed → {PROC_DIR}")

# Patch cfg if already loaded
try:
    cfg["data"]["root"] = PROC_DIR
    print("✓ cfg['data']['root'] updated")
except NameError:
    print("  Note: run cell 5 after this to load cfg")

# ── Verify ──
categories = sorted([
    d for d in os.listdir(PROC_DIR)
    if os.path.isdir(f"{PROC_DIR}/{d}") and d in KNOWN_CATEGORIES
])
print(f"\n✓ Categories found ({len(categories)}): {categories}")

✓ kaggle.json configured
✓ Raw files already present — skipping download

  Contents of RAW_DIR (debug):
raw/
  .ipynb_checkpoints/


RuntimeError: Could not locate MVTec categories. Check tree above.

## 5 · Configuration

Edit this cell to change category, backbone, or PatchCore params.  
Nothing is hardcoded anywhere else — all modules read from this dict.


In [ ]:
import sys
sys.path.insert(0, "/content/anomaly-vision/src")

import yaml
from utils import load_config, override_config, set_seed, get_device, setup_output_dirs

# Load base config
cfg = load_config("/content/anomaly-vision/configs/default.yaml")

# Override for this run — change anything here
cfg = override_config(cfg, {
    "data": {
        "root"      : "/content/anomaly-vision/data/processed",
        "category"  : "bottle",      # try: bottle, carpet, hazelnut, leather ...
        "image_size": 256,
        "batch_size": 8,
    },
    "patchcore": {
        "k_neighbors"    : 9,
        "subsample_ratio": 0.1,
    },
    "diffusion": {
        "noise_level": 0.4,          # 0 = no noise, 1 = full noise (0.3–0.5 is sweet spot)
    },
    "output": {
        "checkpoint_dir": "/content/anomaly-vision/outputs/checkpoints",
        "results_dir"   : "/content/anomaly-vision/outputs/results",
    },
    "training": {"device": "cuda"},
})

set_seed(42)
device = get_device(cfg)
setup_output_dirs(cfg)

print("\n  Config:")
print(f"  Category    : {cfg['data']['category']}")
print(f"  Image size  : {cfg['data']['image_size']}")
print(f"  k-neighbors : {cfg['patchcore']['k_neighbors']}")
print(f"  Subsample   : {cfg['patchcore']['subsample_ratio']}")
print(f"  Noise level : {cfg['diffusion']['noise_level']}")


## 6 · Load dataset

In [ ]:
from dataset import get_dataloaders

train_loader, test_loader = get_dataloaders(cfg)

print(f"  Train (normal only) : {len(train_loader.dataset)} images")
print(f"  Test (all)          : {len(test_loader.dataset)} images")

# Quick sanity check — visualise one training sample
import matplotlib.pyplot as plt
import numpy as np
from utils import denormalize

batch = next(iter(train_loader))
imgs  = batch["image"]

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(denormalize(imgs[i]))
    ax.set_title("normal")
    ax.axis("off")
plt.suptitle(f"Training samples — {cfg['data']['category']}", fontsize=12)
plt.tight_layout()
plt.show()


## 7 · Build memory bank (PatchCore fit)

One forward pass through all normal training images.  
Features are extracted at two scales (layer2 + layer3), fused, then patch-embedded.  
Greedy coreset subsampling reduces the bank to `subsample_ratio` fraction.


In [ ]:
from model import AnomalyDetector

detector = AnomalyDetector(cfg)
detector.fit(train_loader)

print(f"\n  Memory bank size : {detector.memory_bank.bank.shape}")
print(f"  Embedding dim    : {detector.memory_bank.bank.shape[1]}")


## 8 · Inference

Scores every test image.  
PatchCore kNN distance → patch-level score map → upsampled to input resolution.  
Diffusion residual (optional) blended in at 30% weight for pixel-precise refinement.


In [ ]:
USE_DIFFUSION = False   # Set True for diffusion refinement (slower, ~+1 min per category)

image_scores, score_maps, labels = detector.predict(
    test_loader, use_diffusion=USE_DIFFUSION
)

normal_count  = labels.count(0)
anomaly_count = labels.count(1)
print(f"  Scored {len(labels)} images — {normal_count} normal, {anomaly_count} anomalous")


## 9 · Evaluate — AUROC · Pixel AUROC · PRO

In [ ]:
from train    import _collect_gt, _collect_images
from evaluate import evaluate

gt_masks = _collect_gt(test_loader)
images   = _collect_images(test_loader)

results = evaluate(
    image_scores = image_scores,
    score_maps   = score_maps,
    gt_masks     = gt_masks,
    labels       = labels,
    category     = cfg["data"]["category"],
    output_dir   = cfg["output"]["results_dir"],
)


## 10 · Visualise anomaly maps

In [ ]:
from evaluate import save_anomaly_maps
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

save_anomaly_maps(
    images     = images,
    score_maps = score_maps,
    gt_masks   = gt_masks,
    labels     = labels,
    output_dir = cfg["output"]["results_dir"],
    n_samples  = 6,
)

# Display inline
img_path = f"{cfg['output']['results_dir']}/anomaly_maps.png"
fig = plt.figure(figsize=(14, 3 * 6))
plt.imshow(mpimg.imread(img_path))
plt.axis("off")
plt.tight_layout()
plt.show()


## 11 · Save checkpoint

In [ ]:
from train import save_checkpoint
import json

ckpt_dir = f"{cfg['output']['checkpoint_dir']}/{cfg['data']['category']}"
save_checkpoint(detector, cfg, ckpt_dir)

# Also save results json
results_dir = f"{cfg['output']['results_dir']}/{cfg['data']['category']}"
import os; os.makedirs(results_dir, exist_ok=True)
with open(f"{results_dir}/results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n  Saved:")
print(f"  {ckpt_dir}/memory_bank.pkl")
print(f"  {results_dir}/results.json")


## 12 · (Optional) Benchmark — all 15 categories

Runs the full pipeline across every MVTec category and prints a summary table.  
Takes ~30–45 min on T4 without diffusion. Run overnight or skip for now.


In [ ]:
# Uncomment to run full benchmark
# from train import run_all_categories
# all_results = run_all_categories(cfg, use_diffusion=False)
print("Skipped — uncomment above to run full benchmark.")


## 13 · Download results from Colab

In [ ]:
from google.colab import files
import zipfile, os

# Zip outputs and download
zip_path = "/content/anomaly_vision_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk("/content/anomaly-vision/outputs"):
        for fname in fnames:
            fpath = os.path.join(root, fname)
            zf.write(fpath, os.path.relpath(fpath, "/content/anomaly-vision"))

files.download(zip_path)
print("✓ outputs.zip downloaded")
